# Amazon Toys & Games data split

Disjoint global timestamp split: **80% train / 10% validation / 10% test**.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.abspath('../src'))
from preprocess import train_val_test_split, prepare_splitted_data

PROJECT_PATH = os.path.abspath('..')
DATA_PATH = os.path.join(PROJECT_PATH, 'data', 'gg')
os.makedirs(DATA_PATH, exist_ok=True)

RELEVANCE_COL = 'rating'
RELEVANCE_THRESHOLD = 5

In [ ]:
RAW_PATH = os.environ.get(
    'AMAZON_TOYS_PATH',
    os.path.join(PROJECT_PATH, 'data', 'reviews_Toys_and_Games_5.json'),
)

data = pd.read_json(RAW_PATH, lines=True)
data = data[['reviewerID', 'asin', 'overall', 'unixReviewTime']].rename(
    columns={
        'reviewerID': 'user_id',
        'asin': 'item_id',
        'overall': 'rating',
        'unixReviewTime': 'timestamp',
    }
)
data.head()

In [ ]:
print('duplicate user-item pairs:', data.duplicated(['user_id', 'item_id']).sum())
print('negative feedback rate:', np.mean(data[RELEVANCE_COL] < RELEVANCE_THRESHOLD))
train_preview = data  # IDs are re-indexed inside train_val_test_split

In [ ]:
train, val, test = train_val_test_split(
    data,
    RELEVANCE_THRESHOLD,
    RELEVANCE_COL,
    train_quantile=0.8,
    val_quantile=0.9,
)

In [ ]:
print('train users:', train.user_id.nunique())
print('val users:', val.user_id.nunique())
print('test users:', test.user_id.nunique())
train.groupby('user_id')['item_id'].count().describe()

In [ ]:
train.to_parquet(os.path.join(DATA_PATH, 'train.parquet'), index=False)
val.to_parquet(os.path.join(DATA_PATH, 'validation.parquet'), index=False)
test.to_parquet(os.path.join(DATA_PATH, 'test.parquet'), index=False)
print('saved to', DATA_PATH)

In [ ]:
(
    train_p,
    validation_p,
    test_p,
    last_pos_item_test,
    last_pos_item_val,
    last_neg_item_test,
    last_neg_item_val,
) = prepare_splitted_data(
    DATA_PATH,
    relevance_col=RELEVANCE_COL,
    relevance_threshold=RELEVANCE_THRESHOLD,
    verify=True,
)

print('val users with neighbour pair:', last_pos_item_val.user_id.nunique())
print('test users with neighbour pair:', last_pos_item_test.user_id.nunique())